# Binomial Distribution

The **Binomial** distribution counts the number of successes in $n$ independent trials, each with success probability $p$.

| Property | Value |
|----------|-------|
| **Notation** | $X \sim \text{Bin}(n, p)$ |
| **Description** | Number of successes in $n$ independent Bernoulli trials |
| **Parameters** | $n$ = number of trials, $p$ = probability of success |
| **Support** | $\{0, 1, 2, \ldots, n\}$ |
| **PMF** | $P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$ |
| **Expectation** | $E[X] = np$ |
| **Variance** | $\text{Var}(X) = np(1-p)$ |

**Key insight:** $X = \sum_{i=1}^{n} Y_i$ where $Y_i \sim \text{Bern}(p)$ independently.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from math import comb

print("Libraries loaded!")

---
## 1. PMF Derivation — Why $\binom{n}{k}p^k(1-p)^{n-k}$?

For exactly $k$ successes in $n$ trials:
1. **Choose positions:** $\binom{n}{k}$ ways to pick which $k$ trials are successes
2. **Probability of one arrangement:** $p^k (1-p)^{n-k}$
3. **Multiply:** each arrangement is mutually exclusive

In [ ]:
# Visualize PMF for different parameters
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

configs = [
    (10, 0.5, "Fair coin, 10 flips"),
    (10, 0.2, "Biased coin (p=0.2), 10 flips"),
    (50, 0.3, "50 trials, p=0.3"),
    (100, 0.5, "100 trials, p=0.5"),
]

for ax, (n, p, title) in zip(axes.flat, configs):
    rv = stats.binom(n, p)
    x = np.arange(0, n + 1)
    pmf = rv.pmf(x)
    
    # Only plot bars with visible probability
    mask = pmf > 1e-6
    ax.bar(x[mask], pmf[mask], color='#3498db', edgecolor='black', linewidth=0.8, alpha=0.8)
    ax.axvline(rv.mean(), color='red', linewidth=2, linestyle='--',
               label=f'E[X] = {rv.mean():.1f}')
    ax.set_title(f'{title}\nBin(n={n}, p={p})', fontsize=11, fontweight='bold')
    ax.set_xlabel('k (successes)')
    ax.set_ylabel('P(X = k)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 2. Expectation Derivation

Since $X = \sum_{i=1}^{n} Y_i$ where each $Y_i \sim \text{Bern}(p)$:

$$E[X] = E\left[\sum_{i=1}^{n} Y_i\right] = \sum_{i=1}^{n} E[Y_i] = \sum_{i=1}^{n} p = np$$

In [ ]:
# Verify E[X] = np by simulation
np.random.seed(42)

print(f"{'n':>5s} {'p':>5s} | {'Theory':>8s} | {'Simulation':>10s}")
print("-" * 38)
for n, p in [(10, 0.5), (20, 0.3), (50, 0.7), (100, 0.1)]:
    samples = stats.binom.rvs(n, p, size=100000)
    print(f"{n:5d} {p:5.1f} | {n*p:8.2f} | {samples.mean():10.4f}")

---
## 3. Simulation: Binomial as Sum of Bernoullis

In [ ]:
n, p = 20, 0.3
np.random.seed(42)

# Method 1: Direct binomial sampling
direct = stats.binom.rvs(n, p, size=50000)

# Method 2: Sum of Bernoullis
bernoullis = stats.bernoulli.rvs(p, size=(50000, n))
from_bernoullis = bernoullis.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(0, n + 1)
true_pmf = stats.binom.pmf(x, n, p)

for ax, data, title in [(axes[0], direct, "Direct Binom(20, 0.3)"),
                          (axes[1], from_bernoullis, "Sum of 20 Bern(0.3)")]:
    emp = np.array([np.mean(data == k) for k in x])
    ax.bar(x - 0.2, emp, 0.4, color='#e74c3c', alpha=0.7, label='Simulation', edgecolor='black')
    ax.bar(x + 0.2, true_pmf, 0.4, color='#3498db', alpha=0.7, label='Theory', edgecolor='black')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('k')
    ax.set_ylabel('P(X = k)')
    ax.legend()

plt.tight_layout()
plt.show()
print("Both methods give the same distribution ✓")

---
## 4. Python: scipy.stats.binom

In [ ]:
from scipy.stats import binom

n, p = 10, 0.6
rv = binom(n, p)

print(f"Binomial(n={n}, p={p}):")
print(f"  E[X]         = {rv.mean()}")
print(f"  Var(X)       = {rv.var()}")
print(f"  P(X = 3)     = {rv.pmf(3):.4f}")
print(f"  P(X ≤ 5)     = {rv.cdf(5):.4f}")
print(f"  P(X > 7)     = {1 - rv.cdf(7):.4f}")
print(f"  Samples:       {rv.rvs(size=15, random_state=42)}")

# Practical example: defective parts
n_parts, defect_rate = 100, 0.05
X = binom(n_parts, defect_rate)
print(f"\nExample: {n_parts} parts, {defect_rate*100}% defect rate")
print(f"  P(no defects)        = {X.pmf(0):.4f}")
print(f"  P(at most 3 defects) = {X.cdf(3):.4f}")
print(f"  P(more than 10)      = {1 - X.cdf(10):.6f}")
print(f"  Expected defects     = {X.mean()}")

---
## Summary

| Property | Formula |
|----------|---------|
| PMF | $P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$ |
| $E[X]$ | $np$ |
| $\text{Var}(X)$ | $np(1-p)$ |
| Relationship | $\text{Bin}(n,p) = \sum_{i=1}^n \text{Bern}(p)$ |
| Python | `scipy.stats.binom(n, p)` |

**Next:** Poisson Distribution — counting events in a fixed interval.